In [ ]:
# %matplotlib widget

import pandas as pd
import warnings
from pandas import DataFrame
from scipy.signal import filtfilt, butter
import plotly.graph_objects as go
import matplotlib.pyplot as plt
# plt.ion()
import matplotlib.ticker
import matplotlib.dates as mdates
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from parser_utils import folder_selection_utils
from scipy.stats import linregress
import os
import math
import parser_utils.parser_logger
import logging
import json
from parser_utils import folder_selection_utils
from torque_log_analysis import resample_data,drop_rows_from_df,lpf_df
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from motor import Motor

warnings.simplefilter(action='ignore',category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore',category=pd.errors.SettingWithCopyWarning)

plt.style.use('bmh')

# steps


In [ ]:
path = r'C:\Users\Matthew Samson\Documents\formula\6_19_23 ECAR COMP\parsed-data\enduranceOnly'


In [ ]:

for filename in os.listdir(path):
    # print(f"path: {path}")
    if filename.endswith('.csv') or filename.endswith('.CSV'):
        print(f"Filename: {filename}")
        file_path = os.path.join(path, filename)
        df = pd.read_csv(file_path)
        print(list(df))

        df = df[['Time','D1_DC_Bus_Voltage','D1_Commanded_Torque','D2_Motor_Speed','D4_DC_Bus_Current']]
        df = resample_data(df, "Time",resample_interval=100)
        df = lpf_df(df,1)
        df = drop_rows_from_df(df,"D1_Commanded_Torque",1,400)
        df = drop_rows_from_df(df, "D2_Motor_Speed", 10, 7000)

print(list(df))



In [ ]:
# Calculate instantaneous power based on torque command and motor speed
df['instantaneous_power'] = df['D1_Commanded_Torque'] * df['D2_Motor_Speed'] / 9.5488 # watts
df['instantaneous_power'] = df['D1_DC_Bus_Voltage'] * df['D4_DC_Bus_Current']

In [ ]:
from battery import battery,batteryPack
cell= battery(r'battery_soc_curves/lgHE2ahCurve.csv',resistance=32.1/1000)
energus300v = batteryPack(cell,8,72,name='ENERGUS 300v')
print(energus300v.capacity)
print(energus300v.voltage)

In [ ]:
cosmx= battery(r'battery_soc_curves/COSMXahCurve.csv',capacity=13000,resistance=1.5/1000)
cosmx300v = batteryPack(cosmx,2,72,name='COSMX 72s')
cosmx400v = batteryPack(cosmx,1,96,name='COSMX 96s')
cosmx600v= batteryPack(cosmx,1,144,name='COSMX 144s')
cosmx300v.getMaxPowerOut()

In [ ]:
# calc power output vs current
amp_df = pd.DataFrame()
amp_df['amps'] = range(0,1000,10)
amp_df[energus300v.name+'sagv'] = energus300v.getInstantaneousVoltage(amp_df['amps'])
amp_df['power'] = amp_df[energus300v.name+'sagv']*amp_df['amps']
plt.plot(amp_df['amps'],amp_df[energus300v.name+'sagv'])
plt.plot(amp_df['amps'],amp_df['power'])

print(amp_df['power'].max())
print(amp_df['amps'][amp_df['power'].idxmax()])

In [ ]:
# Function to calculate voltage and capacity at each timestamp
previous_row_name = None

def calculate_battery_values_pwr(row,accumulator:batteryPack):
    global previous_row_name

    # Ignore rows with negative power values
    if row['instantaneous_power'] < 0:
        return pd.Series([np.nan, np.nan])

    # Initialize the battery if previous_row_name is None
    if previous_row_name is None:
        previous_row_name = row.name
        return pd.Series([accumulator.getInstantaneousVoltage(0), accumulator.capacity])
    else:
        # Calculate the time difference
        time_diff = row.name - previous_row_name
        # Assuming the battery is discharged at a constant power for each row's duration
        # print(row)
        power = row['instantaneous_power']
        print(f"Power {power}")
        last_voltage = 0
        current = 0
        current_voltage = accumulator.voltage
        while last_voltage != current_voltage:
            last_voltage = current_voltage
            current = power / current_voltage
            current_voltage = accumulator.getInstantaneousVoltage(current)
            print(f"Last voltage: {last_voltage} CUrrent voltage: {current_voltage} Power: {power} CUrent : {current}")
            
        # print(time_diff.total_seconds())
        accumulator.discharge(current, time_diff.total_seconds())

        # Save the current timestamp for the next iteration
        previous_row_name = row.name

    # Return the calculated voltage and capacity
    return pd.Series([accumulator.getInstantaneousVoltage(current), accumulator.capacity,current])

df[['simbattery_voltage', 'simbattery_capacity','simbattery_current']] = df.apply(calculate_battery_values_pwr,accumulator=energus300v, axis=1)

print(df)

In [ ]:
print(cosmx300v.capacity)

In [ ]:
plt.style.use('bmh')
fig, (ax1,ax2) = plt.subplots(2,1,figsize=(16,9))
ax1.plot(df.index,df['simbattery_voltage']*df['simbattery_current']/1000,label='Simulated Battery Power',color='darkblue')
# ax1.plot(df.index,df['D1_DC_Bus_Voltage']*df['D4_DC_Bus_Current']/1000,label='Real Battery Power',color='crimson')

ax2.plot(df.index,df['D1_DC_Bus_Voltage']*df['D4_DC_Bus_Current']/1000,label='Real Battery Power',color='crimson')
axes=plt.gcf().get_axes()
axes[0].title.set_text(("Dumb Battery Model Power, Real Battery Power vs. Time\n2023 Comp Endurance Data")
)
for axis in axes:
    axis.legend(loc='upper right')
    axis.set_xlabel('Time')
    axis.set_ylabel('Power (Kilowatts)')
    # axis.set_ylim((175,325))
    axis.yaxis.set_minor_locator(MultipleLocator(1000))
    axis.yaxis.grid(True,which='minor')

plt.show()

In [ ]:
plt.style.use('bmh')
fig, (ax1) = plt.subplots(1,1,figsize=(16,9))
ax1.plot(df.index,abs((df['simbattery_voltage']*df['simbattery_current']) - (df['D1_DC_Bus_Voltage']*df['D4_DC_Bus_Current'])),label='Simulated - Real Battery Power',color='darkblue')
axes=plt.gcf().get_axes()
axes[0].title.set_text(("Dumb Battery Model Power, Real Battery Power vs. Time\n2023 Comp Endurance Data")
)
for axis in axes:
    axis.legend(loc='upper right')
    axis.set_xlabel('Time')
    axis.set_ylabel('Power (Kilowatts)')
    # axis.set_ylim((175,325))
    axis.yaxis.set_minor_locator(MultipleLocator(1000))
    axis.yaxis.grid(True,which='minor')

plt.show()

In [ ]:
plt.style.use('bmh')
fig, (ax1,ax2) = plt.subplots(2,1,figsize=(16,9))
ax1.plot(df.index,df['simbattery_voltage'],label='Simulated Battery Voltage',color='darkblue')
ax1.plot(df.index,df['D1_DC_Bus_Voltage'],label='D1_DC_Bus_Voltage',color='crimson')
ax1.set_ylim((220,305))
# ax1.plot(df.index,df['simbattery_current'],label='Simulated Battery Current',color='cyan')

ax2.plot(df.index,df['D1_DC_Bus_Voltage'],label='D1_DC_Bus_Voltage',color='crimson')
ax2.plot(df.index,df['D4_DC_Bus_Current'],label='D4_DC_Bus_Current',color='blue')

axes=plt.gcf().get_axes()
axes[0].title.set_text(("Dumb Battery Model, Real Battery Voltage vs. Time\n2023 Comp Endurance Data")
)
for axis in axes:
    axis.legend(loc='center right')
    axis.set_xlabel('Time')
    axis.set_ylabel('Voltage (Volts)')
    # axis.set_ylim((175,325))
    axis.yaxis.set_minor_locator(MultipleLocator(10))
    axis.yaxis.grid(True,which='minor')

plt.show()

In [ ]:
plt.plot()

In [ ]:


previous_row_name = None


# Function to calculate voltage and capacity at each timestamp
def calculate_battery_values(row):
    global previous_row_name

    # Ignore rows with negative current values
    if row['D4_DC_Bus_Current'] < 0:
        return pd.Series([np.nan, np.nan])

    # Initialize the battery if previous_row_name is None
    if previous_row_name is None:
        previous_row_name = row.name
        return pd.Series([energus300v.getInstantaneousVoltage(0), energus300v.capacity])
    else:
        # Calculate the time difference
        time_diff = row.name - previous_row_name
        # Assuming the battery is discharged at a constant current for each row's duration
        current = row['D4_DC_Bus_Current']
        energus300v.discharge(current, time_diff.total_seconds())

        # Save the current timestamp for the next iteration
        previous_row_name = row.name

    # Return the calculated voltage and capacity
    return pd.Series([energus300v.getInstantaneousVoltage(current), energus300v.capacity])


In [ ]:
fig, (ax1,ax2) = plt.subplots(2,1,figsize=(16,9))
ax1.scatter(df['filtD4_DC_Bus_Current'],df['simbattery_voltage'],label='Simulated Battery Voltage',marker='.',color='darkblue')
ax2.scatter(df['filtD4_DC_Bus_Current'],df['filtD1_DC_Bus_Voltage'],label='D1_DC_Bus_Voltage',marker='.',color='crimson')
axes=plt.gcf().get_axes()
axes[0].title.set_text(("Dumb Battery Model, Real Battery Voltage vs. Current Draw\n2023 Comp Endurance Data")
)
for axis in axes:
    axis.legend(loc='upper right')
    axis.set_xlabel('Current (Amps)')
    axis.set_ylabel('Voltage (Volts)')

plt.show()

In [ ]:
plt.style.use('bmh')
fig, (ax1,ax2) = plt.subplots(2,1,figsize=(16,9))
ax1.plot(df.index,df['simbattery_voltage'],label='Simulated Battery Voltage',color='darkblue')
ax2.plot(df.index,df['filtD1_DC_Bus_Voltage'],label='D1_DC_Bus_Voltage',color='crimson')
axes=plt.gcf().get_axes()
axes[0].title.set_text(("Dumb Battery Model, Real Battery Voltage vs. Time\n2023 Comp Endurance Data")
)
for axis in axes:
    axis.legend(loc='upper right')
    axis.set_xlabel('Time')
    axis.set_ylabel('Voltage (Volts)')
    axis.set_ylim((175,325))
    axis.yaxis.set_minor_locator(MultipleLocator(10))
    axis.yaxis.grid(True,which='minor')

plt.show()